## **TMDB Movie Data Analysis using Spark and APIs**


In [1]:
import os
import time
import json
import sys
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
from pathlib import Path


### *Functions from scripts*

In [2]:
sys.path.append(os.path.abspath("..")) 
from src.tmdb_client import fetch_movie_with_credits
from src.bronze_to_spark import read_bronze_json
import src.silver_transform as st

Let us start the Spark session.

In [3]:
spark = SparkSession.builder.appName("tmdb_lab").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/03 12:03:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. **Fetching the movie data from the API**

In this project, we will follow the medallion workflow to ingest, transform and analyze the movie data for the wanted ids.

In [4]:
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

print(f"The TMDB API Key is fetched")

The TMDB API Key is fetched


In [5]:
# TMDB base URLs and Endpoints
BASE_URL = "https://api.themoviedb.org/3"
MOVIE_ENDPOINT = f"{BASE_URL}/movie"
CREDITS_ENDPOINT = f"{BASE_URL}/movie"

# Movie IDs to fetch
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]

len(movie_ids), movie_ids[:5]


(19, [0, 299534, 19995, 140607, 299536])

### **Bronze Layer:**

#### **Bronze Ingestion**
Let us ingest our raw data as JSON first

In [6]:
PROJECT_ROOT = Path(os.getcwd()).parent
out_dir = PROJECT_ROOT / "data" / "bronze" / "tmdb_raw_json"
out_dir.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]


In [7]:

for mid in movie_ids:
    out_path = out_dir / f"movie_id={mid}.json"

    if out_path.exists():
        print(f"Exists, skipping: {mid}")
        continue

    print(f"Fetching: {mid}")
    bundle = fetch_movie_with_credits(mid, TMDB_API_KEY)

    if bundle is None:
        print(f"Movie with Id: {mid} not found")
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, ensure_ascii=False)

Fetching: 0
[ERROR] Failed: https://api.themoviedb.org/3/movie/0 | HTTPSConnectionPool(host='api.themoviedb.org', port=443): Max retries exceeded with url: /3/movie/0?api_key=9094802cf093a61d48b07e86ef4720c5 (Caused by NameResolutionError("HTTPSConnection(host='api.themoviedb.org', port=443): Failed to resolve 'api.themoviedb.org' ([Errno -3] Temporary failure in name resolution)"))
ID 0: Movie not found. Skipping
Movie with Id: 0 not found
Exists, skipping: 299534
Exists, skipping: 19995
Exists, skipping: 140607
Exists, skipping: 299536
Exists, skipping: 597
Exists, skipping: 135397
Exists, skipping: 420818
Exists, skipping: 24428
Exists, skipping: 168259
Exists, skipping: 99861
Exists, skipping: 284054
Exists, skipping: 12445
Exists, skipping: 181808
Exists, skipping: 330457
Exists, skipping: 351286
Exists, skipping: 109445
Exists, skipping: 321612
Exists, skipping: 260513


In [8]:
len(list(out_dir.glob("*.json"))) # Let us see the number of moies fetched

18

#### **Bronze to Spark**
Now we can read the Bronze JSON into Spark

In [9]:
bronze_path = str(out_dir)
bronze_df = read_bronze_json(spark, bronze_path)

In [10]:
bronze_df.printSchema()

bronze_df.select("movie.id", "movie.title", "credits.id","fetched_at").show(5, truncate=False)

print("Bronze count:", bronze_df.count())


root
 |-- credits: struct (nullable = true)
 |    |-- cast: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- cast_id: long (nullable = true)
 |    |    |    |-- character: string (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |    |    |    |-- gender: long (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- known_for_department: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- order: long (nullable = true)
 |    |    |    |-- original_name: string (nullable = true)
 |    |    |    |-- popularity: double (nullable = true)
 |    |    |    |-- profile_path: string (nullable = true)
 |    |-- crew: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- adult: boolean (nullable = true)
 |    |    |    |-- credit_id: string (nullable = true)
 |

26/02/03 12:04:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze count: 18


All our movies are now stored into a spark data frame

## 2. **Silver Layer: Data Cleaning and Preprocessing**

Since we have our data in a dataframe now we can go ahead and clean it according to what we need from it
### **Data Preparation and Cleaning**
#### **Cast and Director Information**
Let us make sure we also have the credits information like crew size and all that.


In [11]:
credits_df = (bronze_df.select( F.col("movie.id").alias("id"),
        F.col("credits.cast").alias("cast_arr"),
        F.col("credits.crew").alias("crew_arr"),))

credits_df = st.add_credits_features(credits_df, cast_top_n=30)

# drop arrays after deriving features
credits_df = credits_df.drop("cast_arr", "crew_arr")

credits_df.show(5, truncate=False)
credits_df.printSchema()

+------+-------------+---------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id    |director     |cast_size|crew_size|cast                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
+-----

#### **Droping irrelevant columns**
Rather than removing unnecessary fields after loading the data, we applied column projection at read time by explicitly selecting only the attributes required for analysis. Columns such as `adult`, `imdb_id`, `original_title`, `video`, and `homepage` were excluded by not selecting them from the nested TMDB JSON structure.


In [12]:
# Keeping only the columns we need (Droping everything else)
base = st.build_silver(bronze_df)
base = base.join(credits_df, on="id", how="left")


In [13]:
base.select("id", "title", "director", "cast_size", "crew_size").show(5, truncate=False)

+------+-----------------------+-------------+---------+---------+
|id    |title                  |director     |cast_size|crew_size|
+------+-----------------------+-------------+---------+---------+
|19995 |Avatar                 |James Cameron|67       |991      |
|299536|Avengers: Infinity War |Anthony Russo|69       |734      |
|24428 |The Avengers           |Joss Whedon  |112      |642      |
|99861 |Avengers: Age of Ultron|Joss Whedon  |74       |654      |
|299534|Avengers: Endgame      |Anthony Russo|107      |608      |
+------+-----------------------+-------------+---------+---------+
only showing top 5 rows


#### **Evaluating JSON-like columns**
TMDB returns several attributes as nested JSON objects (structs) or lists of objects (arrays of structs) rather than flat scalar values. We inspected their Spark schema using `printSchema()` to understand their internal structure before transforming them into analysis-friendly columns.

In [14]:
base.select(
    "belongs_to_collection_raw",
    "genres_raw",
    "spoken_languages_raw",
    "production_countries_raw",
    "production_companies_raw"
).printSchema()

root
 |-- belongs_to_collection_raw: struct (nullable = true)
 |    |-- backdrop_path: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- poster_path: string (nullable = true)
 |-- genres_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- spoken_languages_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- english_name: string (nullable = true)
 |    |    |-- iso_639_1: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- production_countries_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- iso_3166_1: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- production_companies_raw: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- id: long (nullab

From the schema:

- `belongs_to_collection_raw` is a struct containing fields such as `id` and `name` → we extract `name`.

- `genres_raw`, `production_countries_raw`, and `production_companies_raw` are arrays of structs where the main descriptive field is `name`.

- `spoken_languages_raw` is an array of structs that includes both `name` and `english_name` → we typically use `english_name` for consistent readability.

This schema inspection step ensures we extract the correct fields and avoid null/incorrect transformations when flattening the data into Silver format.


#### **Extraction of key Data Points**
Now that we understand the stracture, we can go ahead and extract the data we actually need from the JSON.

**Collection Name**

In [15]:
silver_df = st.add_collection(base)

In [16]:
silver_df.select("id", "title", "belongs_to_collection").show(5, truncate=False)

st.value_counts(silver_df, "belongs_to_collection").show(truncate=False)


+------+-----------------------+-----------------------+
|id    |title                  |belongs_to_collection  |
+------+-----------------------+-----------------------+
|19995 |Avatar                 |Avatar Collection      |
|299536|Avengers: Infinity War |The Avengers Collection|
|24428 |The Avengers           |The Avengers Collection|
|99861 |Avengers: Age of Ultron|The Avengers Collection|
|299534|Avengers: Endgame      |The Avengers Collection|
+------+-----------------------+-----------------------+
only showing top 5 rows
+-----------------------------------+-----+
|belongs_to_collection              |count|
+-----------------------------------+-----+
|The Avengers Collection            |4    |
|Jurassic Park Collection           |2    |
|Star Wars Collection               |2    |
|Frozen Collection                  |2    |
|NULL                               |2    |
|Avatar Collection                  |1    |
|Black Panther Collection           |1    |
|The Incredibles Collec

**Genres**

In [17]:
silver_df = st.add_genres(silver_df)

silver_df.select("id", "title", "genres").show(5, truncate=False)

+------+-----------------------+----------------------------------------+
|id    |title                  |genres                                  |
+------+-----------------------+----------------------------------------+
|19995 |Avatar                 |Action|Adventure|Fantasy|Science Fiction|
|299536|Avengers: Infinity War |Adventure|Action|Science Fiction        |
|24428 |The Avengers           |Science Fiction|Action|Adventure        |
|99861 |Avengers: Age of Ultron|Action|Adventure|Science Fiction        |
|299534|Avengers: Endgame      |Adventure|Science Fiction|Action        |
+------+-----------------------+----------------------------------------+
only showing top 5 rows


In [18]:
silver_df.select( F.explode(F.split(F.col("genres"), "\\|")).alias("genre")).groupBy("genre").count().orderBy(F.desc("count")).show(10)

+---------------+-----+
|          genre|count|
+---------------+-----+
|      Adventure|   15|
|         Action|   12|
|Science Fiction|   10|
|        Fantasy|    5|
|         Family|    5|
|      Animation|    4|
|       Thriller|    3|
|        Romance|    2|
|          Drama|    2|
|          Crime|    1|
+---------------+-----+
only showing top 10 rows


In [19]:
# Value counts to check if all rows are accounted for
st.value_counts(silver_df, "genres").show(truncate=False)


+-----------------------------------------+-----+
|genres                                   |count|
+-----------------------------------------+-----+
|Adventure|Action|Science Fiction         |3    |
|Action|Adventure|Science Fiction         |2    |
|Action|Adventure|Science Fiction|Thriller|2    |
|Action|Adventure|Fantasy|Science Fiction |1    |
|Science Fiction|Action|Adventure         |1    |
|Adventure|Science Fiction|Action         |1    |
|Drama|Romance                            |1    |
|Animation|Family|Adventure|Fantasy       |1    |
|Action|Adventure|Animation|Family        |1    |
|Action|Crime|Thriller                    |1    |
|Family|Fantasy|Romance                   |1    |
|Adventure|Fantasy                        |1    |
|Adventure|Drama|Family|Animation         |1    |
|Family|Animation|Adventure|Comedy|Fantasy|1    |
+-----------------------------------------+-----+



**Spoken Languages**

In [20]:
silver_df = st.add_spoken_languages(silver_df)

silver_df.select("id", "title", "spoken_languages").show(5, truncate=False)

+------+-----------------------+----------------------+
|id    |title                  |spoken_languages      |
+------+-----------------------+----------------------+
|19995 |Avatar                 |English|Spanish       |
|299536|Avengers: Infinity War |English|Xhosa         |
|24428 |The Avengers           |English|Hindi|Russian |
|99861 |Avengers: Age of Ultron|English               |
|299534|Avengers: Endgame      |English|Japanese|Xhosa|
+------+-----------------------+----------------------+
only showing top 5 rows


In [21]:
silver_df.select(F.explode(F.split(F.col("spoken_languages"), "\\|")).alias("language")
).groupBy("language").count().orderBy(F.desc("count")).show()

+--------+-----+
|language|count|
+--------+-----+
| English|   18|
| Russian|    3|
|   Xhosa|    3|
| Spanish|    2|
|  French|    2|
|   Hindi|    1|
|Japanese|    1|
|  Korean|    1|
| Swahili|    1|
| Italian|    1|
|  German|    1|
| Swedish|    1|
|    Thai|    1|
|  Arabic|    1|
+--------+-----+



In [22]:
# Value Counts
st.value_counts(silver_df, "spoken_languages").show(truncate=False)


+---------------------------------------------+-----+
|spoken_languages                             |count|
+---------------------------------------------+-----+
|English                                      |9    |
|English|Spanish                              |1    |
|English|Xhosa                                |1    |
|English|Hindi|Russian                        |1    |
|English|Japanese|Xhosa                       |1    |
|English|Korean|Swahili|Xhosa                 |1    |
|English|Russian                              |1    |
|English|French|German|Swedish|Italian|Russian|1    |
|Arabic|English|Spanish|Thai                  |1    |
|English|French                               |1    |
+---------------------------------------------+-----+



**Production Countries**

In [23]:
silver_df = st.add_production_countries(silver_df)

silver_df.select("id", "title", "production_countries").show(5, truncate=False)



+------+-----------------------+---------------------------------------+
|id    |title                  |production_countries                   |
+------+-----------------------+---------------------------------------+
|19995 |Avatar                 |United States of America|United Kingdom|
|299536|Avengers: Infinity War |United States of America               |
|24428 |The Avengers           |United States of America               |
|99861 |Avengers: Age of Ultron|United States of America               |
|299534|Avengers: Endgame      |United States of America               |
+------+-----------------------+---------------------------------------+
only showing top 5 rows


In [24]:
silver_df.select(F.explode(F.split(F.col("production_countries"), "\\|")).alias("country")
).groupBy("country").count().orderBy(F.desc("count")).show()

+--------------------+-----+
|             country|count|
+--------------------+-----+
|United States of ...|   18|
|      United Kingdom|    2|
+--------------------+-----+



In [25]:
# Value Counts
st.value_counts(silver_df, "production_countries").show(truncate=False)

+---------------------------------------+-----+
|production_countries                   |count|
+---------------------------------------+-----+
|United States of America               |16   |
|United States of America|United Kingdom|1    |
|United Kingdom|United States of America|1    |
+---------------------------------------+-----+



**Production Companies**

In [26]:
silver_df = st.add_production_companies(silver_df)

silver_df.select("id", "title", "production_companies").show(5, truncate=False)


+------+-----------------------+------------------------------------------------------------------------------------+
|id    |title                  |production_companies                                                                |
+------+-----------------------+------------------------------------------------------------------------------------+
|19995 |Avatar                 |Dune Entertainment|Lightstorm Entertainment|20th Century Fox|Ingenious Film Partners|
|299536|Avengers: Infinity War |Marvel Studios                                                                      |
|24428 |The Avengers           |Marvel Studios                                                                      |
|99861 |Avengers: Age of Ultron|Marvel Studios                                                                      |
|299534|Avengers: Endgame      |Marvel Studios                                                                      |
+------+-----------------------+------------------------

In [27]:
silver_df.select(F.explode(F.split(F.col("production_companies"), "\\|")).alias("company")
).groupBy("company").count().orderBy(F.desc("count")).show(10)

+--------------------+-----+
|             company|count|
+--------------------+-----+
|      Marvel Studios|    5|
|  Universal Pictures|    3|
|Lightstorm Entert...|    2|
|    20th Century Fox|    2|
|      Lucasfilm Ltd.|    2|
|Amblin Entertainment|    2|
|Walt Disney Anima...|    2|
|Walt Disney Pictures|    2|
|  Dune Entertainment|    1|
|Ingenious Film Pa...|    1|
+--------------------+-----+
only showing top 10 rows


In [28]:
# Value Counts
st.value_counts(silver_df, "production_companies").show(truncate=False)

+------------------------------------------------------------------------------------+-----+
|production_companies                                                                |count|
+------------------------------------------------------------------------------------+-----+
|Marvel Studios                                                                      |5    |
|Walt Disney Animation Studios                                                       |2    |
|Dune Entertainment|Lightstorm Entertainment|20th Century Fox|Ingenious Film Partners|1    |
|Amblin Entertainment|Universal Pictures|Legendary Pictures                          |1    |
|Amblin Entertainment|Universal Pictures                                             |1    |
|Lucasfilm Ltd.|Bad Robot                                                            |1    |
|Pixar                                                                               |1    |
|Paramount Pictures|20th Century Fox|Lightstorm Entertainment         

For all the transformed columns, the rows match our original extracted rows. The only column spotted to have Null values is movie collection; `belongs_to_collection`. Now we can go ahead and drop the raw JSON columns.

In [29]:
silver_df = silver_df.drop( "belongs_to_collection_raw", "genres_raw",
    "spoken_languages_raw", "production_countries_raw", "production_companies_raw")

silver_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- original_language: string (nullable = true)
 |-- budget: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- runtime: double (nullable = true)
 |-- vote_count: long (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- popularity: double (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- status: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast_size: integer (nullable = true)
 |-- crew_size: integer (nullable = true)
 |-- cast: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- production_companies: string (nullable = true)



### **Handling Missing & Missing Data**

#### **Convert Column Data Types**

Now since we have our columns, we can look at their data types above. All the columns be it numeric, date, or any other columns honestly seem to have their respectful data types.

In [30]:
# A caution for when the data types don't match what we want
silver_df = st.cast_movie_types(silver_df)

silver_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- original_language: string (nullable = true)
 |-- budget: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- runtime: double (nullable = true)
 |-- vote_count: long (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- popularity: double (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- status: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast_size: integer (nullable = true)
 |-- crew_size: integer (nullable = true)
 |-- cast: string (nullable = true)
 |-- belongs_to_collection: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- production_companies: string (nullable = true)



All numeric and temporal fields are explicitly cast to analysis-safe Spark types. This ensures consistency across transformations and prevents downstream schema drift.

#### **Replacing Unrealistic Values**



